# Evaluation & Guardrails: Structured Validation Without LLM-as-Judge

This notebook demonstrates how the Briefcase AI SDK evaluates LLM outputs using **classical NLP guardrails** — not another LLM call. Three distinct `GuardrailEnv` implementations catch different failure modes:

| Guardrail | What It Catches | Method |
|-----------|----------------|--------|
| `FactualAccuracyEnv` | Missing facts, hallucinated entities | ROUGE-L + keyword overlap |
| `ConfidenceCalibrationEnv` | Overconfident or underconfident models | Calibration error analysis |
| `CrossDocConsistencyEnv` | Contradictions across related reports | Entity extraction + comparison |

Each guardrail returns `Effect.ALLOW` or `Effect.DENY` with structured metadata — composable into a `Scorecard` for weighted ranking across model configurations.

> No LLM calls during evaluation. All guardrails are deterministic and fast.

In [ ]:
import sys, os
# Locate the criminal-evidence-workflow root (contains src/ and data/) by walking up
# from the kernel's launch dir, so this works regardless of where it was started.
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not (
    os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'data'))
):
    _root = os.path.dirname(_root)
os.chdir(_root)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import json
from pathlib import Path

import briefcase
from briefcase.guardrails import EvalRequest, Effect
from src.mock_llm import MockLLMProvider
from src.pipeline import summarize_report
from src.config import storage
from src.guardrails.factual_accuracy import FactualAccuracyEnv
from src.guardrails.confidence_calibration import ConfidenceCalibrationEnv
from src.guardrails.consistency import CrossDocConsistencyEnv

print("Guardrail environments loaded:")
for env in [FactualAccuracyEnv, ConfidenceCalibrationEnv, CrossDocConsistencyEnv]:
    print(f"  - {env.__name__}")

## 1. Guardrail #1: Factual Accuracy (ROUGE + Entity Overlap)

This guardrail measures how well the summary captures the source document's key facts. It combines two signals:

- **ROUGE-L** (60% weight): Longest common subsequence between source and summary — measures textual overlap
- **Entity/keyword overlap** (40% weight): Checks whether expected entities from the ground truth appear in the summary

The composite score determines `Effect.ALLOW` (>= 0.5) or `Effect.DENY` (< 0.5).

Let's compare a high-confidence report (report_001, clear burglary) against a low-confidence report (report_003, conflicting witness accounts).

In [ ]:
factual_env = FactualAccuracyEnv()

# --- Report 001: Clean burglary, clear facts ---
report_001 = Path("data/police_reports/report_001.txt").read_text()
gt_001 = json.loads(Path("data/police_reports/report_001_ground_truth.json").read_text())
llm = MockLLMProvider(model="gpt-4o", simulate_latency=False)
result_001 = await summarize_report("report_001", report_001, llm=llm)

accuracy_001 = await factual_env.evaluate(EvalRequest(
    agent="summarizer", action="summarize", resource="report_001",
    context={"source_text": report_001, "summary": result_001["summary"], "ground_truth": gt_001},
))

# --- Report 003: Conflicting witnesses, ambiguous facts ---
report_003 = Path("data/police_reports/report_003.txt").read_text()
gt_003 = json.loads(Path("data/police_reports/report_003_ground_truth.json").read_text())
result_003 = await summarize_report("report_003", report_003, llm=llm)

accuracy_003 = await factual_env.evaluate(EvalRequest(
    agent="summarizer", action="summarize", resource="report_003",
    context={"source_text": report_003, "summary": result_003["summary"], "ground_truth": gt_003},
))

print(f"{'Metric':<22} {'Report 001':>12} {'Report 003':>12}")
print("─" * 48)
print(f"{'ROUGE-L':<22} {accuracy_001.metadata['rouge_l']:>12.3f} {accuracy_003.metadata['rouge_l']:>12.3f}")
print(f"{'Entity overlap':<22} {accuracy_001.metadata['entity_overlap']:>12.3f} {accuracy_003.metadata['entity_overlap']:>12.3f}")
print(f"{'Composite':<22} {accuracy_001.metadata['composite']:>12.3f} {accuracy_003.metadata['composite']:>12.3f}")
print(f"{'Effect':<22} {str(accuracy_001.effect):>12} {str(accuracy_003.effect):>12}")
print(f"\nReport 001 reason: {accuracy_001.reason}")
print(f"Report 003 reason: {accuracy_003.reason}")

## 2. Guardrail #2: Confidence Calibration

A model that reports 0.93 confidence but only achieves 0.44 accuracy is **overconfident** — dangerous in criminal justice, where false certainty can mislead attorneys.

This guardrail computes `|confidence - accuracy|` and flags outputs where the gap exceeds 0.2 (configurable threshold).

In [ ]:
calibration_env = ConfidenceCalibrationEnv(max_calibration_error=0.2)

# Calibrate report_001 (high confidence, moderate accuracy)
cal_001 = await calibration_env.evaluate(EvalRequest(
    agent="summarizer", action="summarize", resource="report_001",
    context={
        "confidence": result_001["confidence"],
        "accuracy_score": accuracy_001.metadata["composite"],
    },
))

# Calibrate report_003 (low confidence, low accuracy — but closer to calibrated!)
cal_003 = await calibration_env.evaluate(EvalRequest(
    agent="summarizer", action="summarize", resource="report_003",
    context={
        "confidence": result_003["confidence"],
        "accuracy_score": accuracy_003.metadata["composite"],
    },
))

print(f"{'Metric':<24} {'Report 001':>12} {'Report 003':>12}")
print("─" * 50)
print(f"{'Model confidence':<24} {cal_001.metadata['confidence']:>12.2f} {cal_003.metadata['confidence']:>12.2f}")
print(f"{'Measured accuracy':<24} {cal_001.metadata['accuracy_score']:>12.2f} {cal_003.metadata['accuracy_score']:>12.2f}")
print(f"{'Calibration error':<24} {cal_001.metadata['calibration_error']:>12.2f} {cal_003.metadata['calibration_error']:>12.2f}")
print(f"{'Direction':<24} {cal_001.metadata['direction']:>12} {cal_003.metadata['direction']:>12}")
print(f"{'Effect':<24} {str(cal_001.effect):>12} {str(cal_003.effect):>12}")

if cal_001.effect == Effect.DENY:
    print(f"\n** Report 001 flagged: {cal_001.reason}")
if cal_003.effect == Effect.ALLOW:
    print(f"\n   Report 003 passes: low confidence correctly reflects low accuracy")

## 3. Guardrail #3: Cross-Document Consistency

Reports 004 and 005 describe thefts at the **same store** (Bay Area Electronics) on different dates. A good summarization system should produce consistent facts across related documents.

This guardrail extracts entities (addresses, store names, people, amounts) from both summaries and checks for contradictions. The Claude Sonnet fixture for report_005 contains an intentional error: "Lakeshore Boulevard" instead of "Lakeshore Avenue."

In [ ]:
consistency_env = CrossDocConsistencyEnv()

# Get summaries for reports 004 and 005 — first with GPT-4o (consistent)
gpt4o = MockLLMProvider(model="gpt-4o", simulate_latency=False)
summary_004_gpt = (await gpt4o.generate("report_004", "Summarize"))["summary"]
summary_005_gpt = (await gpt4o.generate("report_005", "Summarize"))["summary"]

# Then with Claude Sonnet (report_005 has the Boulevard/Avenue error)
claude = MockLLMProvider(model="claude-sonnet", provider="anthropic", simulate_latency=False)
summary_004_claude = (await claude.generate("report_004", "Summarize"))["summary"]
summary_005_claude = (await claude.generate("report_005", "Summarize"))["summary"]

# Check GPT-4o pair
consistency_gpt = await consistency_env.evaluate(EvalRequest(
    agent="summarizer", action="consistency_check",
    resource="report_004+report_005",
    context={"summary_a": summary_004_gpt, "summary_b": summary_005_gpt},
))

# Check Claude pair (should catch the address mismatch)
consistency_claude = await consistency_env.evaluate(EvalRequest(
    agent="summarizer", action="consistency_check",
    resource="report_004+report_005",
    context={"summary_a": summary_004_claude, "summary_b": summary_005_claude},
))

print("=== GPT-4o (reports 004 + 005) ===")
print(f"  Consistency score: {consistency_gpt.metadata['consistency_score']:.2f}")
print(f"  Contradictions:    {consistency_gpt.metadata['contradictions']}")
print(f"  Effect:            {consistency_gpt.effect}")

print(f"\n=== Claude Sonnet (reports 004 + 005) ===")
print(f"  Consistency score: {consistency_claude.metadata['consistency_score']:.2f}")
print(f"  Contradictions:    {consistency_claude.metadata['contradictions']}")
print(f"  Effect:            {consistency_claude.effect}")

if consistency_claude.metadata['contradictions']:
    print(f"\n  ** Claude Sonnet's report_005 summary says 'Lakeshore Boulevard'")
    print(f"     but report_004 says 'Lakeshore Avenue' — guardrail caught it!")

## 4. Scorecard: Weighted Composite Scoring

The `Scorecard` combines all guardrail results into a single weighted score. This is how you rank model configurations and make deployment decisions.

Weights reflect what matters most for criminal evidence:
- **Factual accuracy** (50%): Getting the facts right is non-negotiable
- **Calibration** (30%): The system must know when it doesn't know
- **Consistency** (20%): Related cases must not contradict each other

In [ ]:
from briefcase._native import Scorecard

scorecard = Scorecard()
scorecard = scorecard.add_score("factual_accuracy", accuracy_001.metadata["composite"], 0.5)
scorecard = scorecard.add_score("confidence_calibration", cal_001.metadata["calibration_score"], 0.3)
scorecard = scorecard.add_score("cross_doc_consistency", consistency_gpt.metadata["consistency_score"], 0.2)

print(f"Scorecard for GPT-4o on report_001:")
print(f"{'─' * 50}")
print(f"  Factual accuracy:     {accuracy_001.metadata['composite']:.3f}  (weight: 0.5)")
print(f"  Calibration:          {cal_001.metadata['calibration_score']:.3f}  (weight: 0.3)")
print(f"  Consistency:          {consistency_gpt.metadata['consistency_score']:.3f}  (weight: 0.2)")
print(f"{'─' * 50}")
print(f"  Composite score:      {scorecard.composite_score:.3f}")

## 5. Full Model Comparison: GPT-4o vs Claude Sonnet

Run all 5 reports through both models, evaluate each with all 3 guardrails, and compare.

In [ ]:
from src.evaluation import run_full_evaluation, print_comparison_table

eval_data = await run_full_evaluation(simulate_latency=False)
print_comparison_table(eval_data)

## 6. Deep Dive: Per-Report Confidence vs Accuracy

Let's visualize the calibration landscape across all reports and models. Well-calibrated outputs cluster along the diagonal (confidence == accuracy).

In [ ]:
print(f"\n{'Report':<12} {'Model':<16} {'Confidence':>10} {'Accuracy':>10} {'Gap':>8} {'Calibrated?':>12}")
print("─" * 72)

for r in eval_data["all_results"]:
    gap = abs(r["confidence"] - r["accuracy"])
    calibrated = "yes" if gap < 0.2 else "NO"
    marker = " **" if calibrated == "NO" else ""
    print(
        f"{r['report_id']:<12} {r['model']:<16} "
        f"{r['confidence']:>10.2f} {r['accuracy']:>10.2f} "
        f"{gap:>8.2f} {calibrated:>12}{marker}"
    )

# Count miscalibrated
miscal = sum(1 for r in eval_data["all_results"] if abs(r["confidence"] - r["accuracy"]) >= 0.2)
print(f"\nMiscalibrated outputs: {miscal}/{len(eval_data['all_results'])} (threshold: 0.2)")
print("** These would be flagged by ConfidenceCalibrationEnv in production")

## 7. Cost Tracking

`CostCalculator` computes per-summary cost from token usage — essential for budget planning when scaling to thousands of case files.

In [ ]:
from briefcase.cost import CostCalculator

cost_calc = CostCalculator()

print(f"Available models: {cost_calc.get_available_models()[:6]}...")

# Estimate cost for each result
print(f"\n{'Report':<12} {'Model':<16} {'Prompt Tok':>10} {'Compl Tok':>10}")
print("─" * 52)

for r in eval_data["all_results"]:
    usage = r["token_usage"]
    print(
        f"{r['report_id']:<12} {r['model']:<16} "
        f"{usage['prompt_tokens']:>10,} {usage['completion_tokens']:>10,}"
    )

total_prompt = sum(r["token_usage"]["prompt_tokens"] for r in eval_data["all_results"])
total_completion = sum(r["token_usage"]["completion_tokens"] for r in eval_data["all_results"])
print(f"\n{'TOTAL':<28} {total_prompt:>10,} {total_completion:>10,}")
print(f"{'':28} {'─' * 22}")
print(f"{'Combined':28} {total_prompt + total_completion:>10,} tokens")

## Key Takeaways

- **3 guardrails, zero LLM calls**: Factual accuracy, confidence calibration, and cross-document consistency — all classical NLP
- **Structured evaluation results**: Each guardrail returns `EvalResult` with `Effect`, `reason`, `eval_time_ms`, and typed `metadata`
- **Composable scoring**: `Scorecard` with configurable weights lets you rank model configs by what matters for your domain
- **Catches real issues**: Miscalibrated confidence (report_001), address contradictions (Claude Sonnet report_005), conflicting witness handling (report_003)
- **Cost-aware**: Token tracking built into every `DecisionSnapshot`

**Next:** See [03_error_reproduction_and_triage.ipynb](./03_error_reproduction_and_triage.ipynb) for deterministic replay and confidence-based routing.